In [1]:
import pandas as pd
data_dir = "/datafs_a/carolgao/haim14_data/blood_infection/"

In [2]:
df = pd.read_csv(f"{data_dir}base_HH_raw_notes.csv")

In [3]:
df.head()

,Unnamed: 0.1,Unnamed: 0,PatientDurableKey,primarymrn,EncounterKey,OrderedYear,OrderedDateKey,Sets,Positive,BothPositive,...,rx_Volatile Anesthetics,rx_Water Soluble Vitamins,rx_Wound Care Products,rx_Xanthines,rx_Zinc,count_rx,count_icd,count_cmp,CombinedNotes,Hospital
0,0,31,131848795,2006623046,512973361,2025,20250702,2,0,0,...,0,0,0,0,0,1,1,1,TRAUMA HISTORY & PHYSICAL Patient Name: ...,HH
1,1,32,131848757,2006623049,512973293,2025,20250703,1,0,0,...,0,0,0,0,0,1,0,1,"8.0 ett, 21 at the lip Melissa Johnson, RN...",HH
2,4,43,131820606,2006621658,512336657,2025,20250630,2,0,0,...,0,0,0,0,0,1,1,1,History Chief Complaint Patient presents wi...,HH
3,7,52,131811399,1001734922,380240849,2023,20231111,1,1,1,...,0,2,0,0,0,1,1,1,History Chief Complaint Patient presents wi...,HH
4,8,53,131811399,1001734922,438331938,2024,20240613,2,0,0,...,0,9,0,0,0,1,1,1,"Patient resting on stretcher, appears to be sl...",HH


In [4]:
from notes_truncation import truncate_notes_by_relevance

# Truncate the notes
df["CombinedNotes"] = df["CombinedNotes"].fillna("")
df['TruncatedNotes'] = df['CombinedNotes'].apply(truncate_notes_by_relevance)
df.to_csv(f"{data_dir}notes_matched_all.csv")

In [7]:
df[['EncounterKey','CombinedNotes','TruncatedNotes']].to_csv(f"{data_dir}notes_matched_HH_all_notes.csv")

In [8]:
df[['EncounterKey','CombinedNotes','TruncatedNotes']].head()

,EncounterKey,CombinedNotes,TruncatedNotes
0,512973361,TRAUMA HISTORY & PHYSICAL Patient Name: ...,He lost his access and a right femoral cordis ...
1,512973293,"8.0 ett, 21 at the lip Melissa Johnson, RN...","Skin: Normal for age and race, grossly normal ..."
2,512336657,History Chief Complaint Patient presents wi...,"Denies fever/chills, diarrhea, melena/hematoch..."
3,380240849,History Chief Complaint Patient presents wi...,"He denies fevers, chills, chest pain, shortnes..."
4,438331938,"Patient resting on stretcher, appears to be sl...",Pulmonary: Effort: Pulmonary effort is no...


In [22]:
# Obtain embeddings for truncated notes

from generate_embeddings import generate_embeddings
generate_embeddings(suffix="HH_all_notes")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding notes: 100%|████████████████████| 41064/41064 [16:32<00:00, 41.38it/s]


In [23]:
# load in generated embeddings
embeddings = pd.read_csv(f"{data_dir}blood_processed_notes_TruncatedNotes_HH_all_notes.csv")

In [24]:
embeddings.head()

,EncounterKey,cn_0,cn_1,cn_2,cn_3,cn_4,cn_5,cn_6,cn_7,cn_8,...,cn_758,cn_759,cn_760,cn_761,cn_762,cn_763,cn_764,cn_765,cn_766,cn_767
0,512973361,-0.069960,0.014469,0.842030,-0.999731,0.998955,0.743175,-0.049615,-0.152250,0.214587,...,-0.021169,-0.052127,-0.008430,0.007043,-0.074440,-0.144216,0.008013,0.976831,-0.062571,0.999492
1,512973293,-0.032809,0.017430,0.944214,-0.999819,0.999133,0.479927,-0.072692,0.323228,0.281087,...,-0.109545,0.078427,-0.170140,0.016284,-0.121120,-0.201823,0.044030,0.990831,-0.050428,0.999752
2,512336657,-0.129203,0.022709,0.811261,-0.999717,0.998760,0.569109,-0.119588,0.004055,0.261186,...,-0.027532,-0.131816,-0.167152,0.046168,-0.058796,-0.136905,0.089274,0.966660,-0.158796,0.999555
3,380240849,-0.149935,-0.031266,0.874963,-0.999687,0.998580,0.743611,-0.094669,0.026609,0.263417,...,0.053528,-0.040520,-0.241094,-0.097352,-0.069277,-0.069286,0.036623,0.984851,-0.191315,0.999421
4,438331938,-0.084096,-0.100061,0.330776,-0.999369,0.996979,0.935622,-0.081762,-0.863017,0.098087,...,-0.012319,-0.121511,0.015527,-0.014261,-0.040388,-0.209551,0.031253,0.833764,-0.215157,0.998754


In [25]:
# Merge with original train and test

df_train = pd.read_csv(f"{data_dir}base_HH_emergency_before_2025.csv")
df_test = pd.read_csv(f"{data_dir}base_HH_emergency_2025.csv")
print("before merging: ", df_train.shape, df_test.shape)

notes_cols = [c for c in df_train.columns if c.startswith("cn_")]
df_train = df_train.drop(columns = notes_cols)
df_test = df_test.drop(columns = notes_cols)

df_train = df_train.merge(embeddings, on = "EncounterKey", how = "left")
df_test = df_test.merge(embeddings, on = "EncounterKey", how = "left")
print("after merging: ", df_train.shape, df_test.shape)

before merging:  (31760, 4299) (4744, 4299)
after merging:  (31760, 4299) (4744, 4299)


In [26]:
df_train.to_csv("/datafs_a/stang01/bsi-mmai/data/base_HH_emergency_before_2025_trunc_all_notes.csv")
df_test.to_csv("/datafs_a/stang01/bsi-mmai/data/base_HH_emergency_2025_trunc_all_notes.csv")